
# Inverse APJN overlays for four `(attn_mult, mlp_mult)` cases

This notebook:

1. Loads **four inverse-fit bundles**.
2. Computes inverse APJN curves from the recursion code for each case.
3. Computes the inverse asymptotic curves
   \[
   \log J_{\mathrm{derf}}^{L,l} \approx \frac{\sqrt{L}-\sqrt{l}}{\sqrt{\lambda}},
   \qquad
   J_{\mathrm{preLN}}^{L,l} \approx \left(\frac{L}{l}\right)^\zeta.
   \]
4. Makes one figure with:
   - a **left 2×2 grid** for inverse derf,
   - a **right 2×2 grid** for inverse pre-LN,

   arranged as

   - upper-left: `attn=2, mlp=1`
   - upper-right: `attn=2, mlp=2`
   - lower-left: `attn=1, mlp=1`
   - lower-right: `attn=1, mlp=2`

You can edit the bundle paths and the `(q0, p0)` values for each case independently in the configuration cell below.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

import math
import pickle
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, LinearSegmentedColormap
from matplotlib.cm import ScalarMappable

mpl.rcParams["figure.dpi"] = 130
mpl.rcParams["savefig.dpi"] = 220
mpl.rcParams["axes.spines.top"] = False
mpl.rcParams["axes.spines.right"] = False
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.alpha"] = 0.18
mpl.rcParams["grid.linewidth"] = 0.6

EPS = 1e-12
LOG_EPS = 1e-300
PI = np.pi


In [ ]:

# -----------------------------------------------------------------------------
# Main user configuration
# -----------------------------------------------------------------------------

BASE_DIR = Path("/content/drive/MyDrive/ml_projects/mapes_variance")

def default_bundle_path(attn_mult, mlp_mult):
    return str(
        BASE_DIR
        / f"inverse_points_depth128_nlayers16_asymp_seed_124_num_inits_4_attn_mult_{attn_mult:.1f}_mlp_mult_{mlp_mult:.1f}"
        / "results.pkl"
    )

CASE_CONFIGS = [
    {
        "label": "attn=2, mlp=1",
        "attn_mult": 2.0,
        "mlp_mult": 1.0,
        "bundle_path": default_bundle_path(2.0, 1.0),
        "q0": 0.27,
        "p0": 0.11,
    },
    {
        "label": "attn=2, mlp=2",
        "attn_mult": 2.0,
        "mlp_mult": 2.0,
        "bundle_path": default_bundle_path(2.0, 2.0),
        "q0": 0.27,
        "p0": 0.11,
    },
    {
        "label": "attn=1, mlp=1",
        "attn_mult": 1.0,
        "mlp_mult": 1.0,
        "bundle_path": default_bundle_path(1.0, 1.0),
        "q0": 0.27,
        "p0": 0.11,
    },
    {
        "label": "attn=1, mlp=2",
        "attn_mult": 1.0,
        "mlp_mult": 2.0,
        "bundle_path": default_bundle_path(1.0, 2.0),
        "q0": 0.27,
        "p0": 0.11,
    },
]

# The recursion signature keeps n_tokens and sigma_A, even though sigma_A is not
# used inside simulate_recursions_full below.
N_TOKENS = 197

# Base values from the prompt.
BASE_SIGMA_W1 = 0.64 * math.sqrt(768 / 1024)
BASE_SIGMA_W2 = 1.28 * math.sqrt(768 / 1024)
BASE_SIGMA_O  = 0.64 * math.sqrt(768 / 1024)
BASE_SIGMA_V  = 0.64 * math.sqrt(768 / 1024)
BASE_SIGMA_A  = 0.64 * 0.64 * 768 / 1024

# Optional toggle:
# False -> use sigma_OV = sigma_O * sigma_V   (matches the recursion scales)
# True  -> use sigma_OV = sigma_O * sigma_V * sigma_A
#         (kept only as an optional switch because a later section of the uploaded
#          file uses that convention)
USE_SIGMA_A_IN_ASYMPTOTICS = False

# Plot styling
DERF_POINT_ALPHA = 0.20
PRELN_POINT_ALPHA = 0.20
DERF_POINT_SIZE = 16
PRELN_POINT_SIZE = 16
DERF_LINEWIDTH = 2.0
PRELN_LINEWIDTH = 2.0
ASYM_LINEWIDTH = 2.0
ASYM_LINESTYLE = "--"

# Save figure if desired
SAVE_FIGURE = False
FIGURE_PATH = "inverse_apjn_four_case_tiles.png"



## Configuration

Edit this cell before running:
- `BASE_DIR` if needed,
- each bundle path if your directory layout differs,
- `(q0, p0)` for each of the four cases.


In [ ]:

# -----------------------------------------------------------------------------
# Helpers from / adapted to the current theory code
# -----------------------------------------------------------------------------

def _safe_divide(a, b, eps=1e-12):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    return a / np.maximum(b, eps)

def kappa_relu_np(rho):
    rho = np.clip(rho, -1.0, 1.0)
    return (1.0 / (2.0 * np.pi)) * (
        np.sqrt(np.maximum(0.0, 1.0 - rho**2)) + rho * (np.pi - np.arccos(rho))
    )

def tilde_q_erf_np(q, alpha):
    x = (2.0 * alpha**2 * q) / (1.0 + 2.0 * alpha**2 * q)
    x = np.clip(x, -1.0, 1.0)
    return (2.0 / np.pi) * np.arcsin(x)

def tilde_p_erf_np(q, p, alpha):
    x = (2.0 * alpha**2 * p) / (1.0 + 2.0 * alpha**2 * q)
    x = np.clip(x, -1.0, 1.0)
    return (2.0 / np.pi) * np.arcsin(x)

def simulate_recursions_full(
    num_layers: int,
    p0: float,
    n_tokens: int,
    mode: str,
    alpha: float = 1.0,
    sigma_w1: float = 0.64,
    sigma_w2: float = 1.28,
    sigma_o: float = 0.64,
    sigma_v: float = 0.64,
    sigma_a: float = 0.64 * 0.64,
    q0: float = 1.0,
    mask_all_p_values: bool = False,
):
    L = int(num_layers)
    n = float(n_tokens)
    eps = 1e-12

    q = np.zeros(L + 1, dtype=float)
    p = np.zeros(L + 1, dtype=float)
    chi_att = np.zeros(L, dtype=float)
    chi_mlp = np.zeros(L, dtype=float)
    q_hat_arr = np.zeros(L, dtype=float)
    p_hat_arr = np.zeros(L, dtype=float)
    q_hat_half_arr = np.zeros(L, dtype=float)
    p_hat_half_arr = np.zeros(L, dtype=float)
    kappa_prime_arr = np.zeros(L, dtype=float)
    K_direct = np.zeros(L + 1, dtype=float)
    J_direct = np.zeros(L + 1, dtype=float)

    q[0], p[0] = float(q0), float(0.0 if mask_all_p_values else p0)
    K_direct[0] = 0.0
    J_direct[0] = 1.0

    att_scale = (sigma_o ** 2) * (sigma_v ** 2)
    mlp_scale = (sigma_w1 ** 2) * (sigma_w2 ** 2)

    for l in range(L):
        ql, pl = q[l], p[l]

        if mode.lower() == "erf":
            denom = (1.0 + 2.0 * alpha**2 * ql) ** 2 - 4.0 * (alpha**4) * (pl**2)
            p_hat = (4.0 * alpha**2 / np.pi) * (
                1.0 / np.sqrt(np.maximum(denom, eps))
            )
            q_hat = (4.0 * alpha**2 / np.pi) * (
                1.0 / np.sqrt(np.maximum(1.0 + 4.0 * alpha**2 * ql, eps))
            )

            uq = tilde_q_erf_np(ql, alpha)
            up = tilde_p_erf_np(ql, pl, alpha)

            beta = np.exp((sigma_a ** 2) * uq * (up - uq))
            gamma = np.exp((sigma_a ** 2) * up * (up - uq))
            q_mix = (uq + (n - 1.0) * up * beta) / (1.0 + (n - 1.0) * beta)
            p_mix = (uq + (n - 1.0) * up * gamma) / (1.0 + (n - 1.0) * gamma)
            if mask_all_p_values:
                q_half = ql + att_scale * q_mix
                p_half = 0.0
            else:
                q_half = ql + att_scale * up
                p_half = pl + att_scale * up

            denom_half = (1.0 + 2.0 * alpha**2 * q_half) ** 2 - 4.0 * (alpha**4) * (p_half**2)
            p_hat_half = (4.0 * alpha**2 / np.pi) * (
                1.0 / np.sqrt(np.maximum(denom_half, eps))
            )
            q_hat_half = (4.0 * alpha**2 / np.pi) * (
                1.0 / np.sqrt(np.maximum(1.0 + 4.0 * alpha**2 * q_half, eps))
            )

            u_half = tilde_q_erf_np(q_half, alpha)
            v_half = tilde_p_erf_np(q_half, p_half, alpha)
            rho_half = np.clip(v_half / (u_half + eps), -1.0, 1.0)

            dq_mlp = 0.5 * mlp_scale * u_half
            dp_mlp = 0.0 if mask_all_p_values else mlp_scale * u_half * kappa_relu_np(rho_half)

            q[l + 1] = q_half + dq_mlp
            p[l + 1] = 0.0 if mask_all_p_values else p_half + dp_mlp

        elif mode.lower() == "layernorm":
            p_hat = 1.0 / (ql + eps)
            q_hat = 1.0 / (ql + eps)

            rho = np.clip(pl / (ql + eps), -1.0, 1.0)

            beta = np.exp((sigma_a ** 2) * (rho - 1.0))
            gamma = np.exp((sigma_a ** 2) * rho * (rho - 1.0))
            q_mix = (1.0 + (n - 1.0) * rho * beta) / (1.0 + (n - 1.0) * beta)
            p_mix = (1.0 + (n - 1.0) * rho * gamma) / (1.0 + (n - 1.0) * gamma)
            if mask_all_p_values:
                q_half = ql + att_scale * q_mix
                p_half = 0.0
            else:
                q_half = ql + att_scale * rho
                p_half = pl + att_scale * rho

            q_hat_half = 1.0 / (q_half + eps)
            p_hat_half = 1.0 / (q_half + eps)

            rho_half = np.clip(p_half / (q_half + eps), -1.0, 1.0)
            dq_mlp = 0.5 * mlp_scale
            dp_mlp = 0.0 if mask_all_p_values else mlp_scale * kappa_relu_np(rho_half)

            q[l + 1] = q_half + dq_mlp
            p[l + 1] = 0.0 if mask_all_p_values else p_half + dp_mlp
        else:
            raise ValueError("mode must be 'erf' or 'layernorm'")

        kappa_prime = 0.25 + (1.0 / (2.0 * np.pi)) * np.arcsin(np.clip(rho_half, -1.0, 1.0))
        q_hat_arr[l] = q_hat
        p_hat_arr[l] = p_hat
        q_hat_half_arr[l] = q_hat_half
        p_hat_half_arr[l] = p_hat_half
        kappa_prime_arr[l] = kappa_prime

        chi_att[l] = 1.0 + att_scale * p_hat
        chi_mlp[l] = 1.0 + 0.5 * mlp_scale * q_hat_half

        J_half = (1 + att_scale * q_hat / n) * J_direct[l] + att_scale * p_hat * K_direct[l]
        K_direct_half = (1 + att_scale * p_hat) * K_direct[l] + (att_scale * q_hat / n) * J_direct[l]

        K_direct[l + 1] = (1.0 + mlp_scale * kappa_prime * p_hat_half) * K_direct_half
        J_direct[l + 1] = chi_mlp[l] * J_half

    chi = chi_att * chi_mlp
    J = np.zeros(L + 1, dtype=float)
    K = np.zeros(L + 1, dtype=float)
    J[L] = 1.0
    K[L] = 0.0
    for l in range(L - 1, -1, -1):
        J_half = (1.0 + 0.5 * mlp_scale * q_hat_half_arr[l]) * J[l + 1]
        K_half = (1.0 + mlp_scale * kappa_prime_arr[l] * p_hat_half_arr[l]) * K[l + 1]
        J[l] = (1.0 + (att_scale / n) * q_hat_arr[l]) * J_half + att_scale * q_hat_arr[l] * K_half
        K[l] = (1.0 + att_scale * p_hat_arr[l]) * K_half + (att_scale / n) * p_hat_arr[l] * J_half

    return {
        "q": q,
        "p": p,
        "p_over_q": _safe_divide(p, q),
        "chi_att": chi_att,
        "chi_mlp": chi_mlp,
        "chi": chi,
        "K": K,
        "K_direct": K_direct,
        "J": J,
        "J_direct": J_direct,
    }

def p_of_c(c):
    c = np.clip(c, 0.0, 1.0)
    return (2.0 / np.pi) * np.arcsin(c)

def kappa(x):
    x = np.clip(x, -1.0, 1.0)
    return (1.0 / (2.0 * np.pi)) * (
        np.sqrt(np.maximum(0.0, 1.0 - x**2)) + x * (np.pi - np.arccos(x))
    )

def bisect_root(fun, a, b, tol=1e-12, max_iter=200):
    fa, fb = fun(a), fun(b)
    if np.isnan(fa) or np.isnan(fb):
        raise ValueError("NaN encountered at bracket endpoints.")
    if fa == 0.0:
        return a
    if fb == 0.0:
        return b
    if fa * fb > 0:
        raise ValueError("Bisection requires a sign change on [a, b].")

    lo, hi = a, b
    flo, fhi = fa, fb
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        fmid = fun(mid)
        if abs(fmid) <= tol or (hi - lo) <= tol:
            return mid
        if flo * fmid <= 0:
            hi, fhi = mid, fmid
        else:
            lo, flo = mid, fmid
    return 0.5 * (lo + hi)

def find_p_star(sigma_OV, sigma_21, grid_n=4001):
    def g(c):
        pc = p_of_c(c)
        num = (sigma_OV**2) * pc + (sigma_21**2) * kappa(pc)
        den = (sigma_OV**2) * pc + (sigma_21**2) / 2.0
        return num / den

    def f(c):
        return g(c) - c

    xs = np.linspace(0.0, 1.0, grid_n)
    fs = np.array([f(x) for x in xs], dtype=float)

    roots = []

    near = np.where(np.abs(fs) < 1e-8)[0]
    for idx in near:
        roots.append(xs[idx])

    for i in range(grid_n - 1):
        a, b = xs[i], xs[i + 1]
        fa, fb = fs[i], fs[i + 1]
        if fa == 0.0:
            roots.append(a)
        if fa * fb < 0:
            roots.append(bisect_root(f, a, b, tol=1e-12, max_iter=200))

    roots = np.array(sorted(roots), dtype=float)
    dedup = []
    for r in roots:
        if not dedup or abs(r - dedup[-1]) > 1e-7:
            dedup.append(r)
    roots = np.array(dedup, dtype=float)

    if roots.size == 0:
        best = xs[np.argmin(np.abs(fs))]
        roots = np.array([best], dtype=float)

    cand = roots[roots < (1.0 - 1e-10)]
    if cand.size > 0:
        c_star = float(np.max(cand))
    else:
        c_star = float(xs[xs < 1.0][-1])

    p_star = float(p_of_c(c_star))
    return p_star, c_star, roots


In [ ]:

# -----------------------------------------------------------------------------
# Bundle loading / extraction helpers
# -----------------------------------------------------------------------------

def load_pickle(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Bundle not found: {path}")
    with path.open("rb") as f:
        return pickle.load(f)

def unwrap_fit_bundle(obj):
    if isinstance(obj, dict) and "fit_bundle" in obj and isinstance(obj["fit_bundle"], dict):
        return obj["fit_bundle"]
    return obj

def get_case_sigmas(attn_mult, mlp_mult):
    sigma_w1 = BASE_SIGMA_W1 * float(mlp_mult)
    sigma_w2 = BASE_SIGMA_W2 * float(mlp_mult)
    sigma_o = BASE_SIGMA_O * float(attn_mult)
    sigma_v = BASE_SIGMA_V * float(attn_mult)
    sigma_a = BASE_SIGMA_A

    sigma_21 = sigma_w1 * sigma_w2
    sigma_OV = sigma_o * sigma_v

    return {
        "sigma_w1": sigma_w1,
        "sigma_w2": sigma_w2,
        "sigma_o": sigma_o,
        "sigma_v": sigma_v,
        "sigma_a": sigma_a,
        "sigma_21": sigma_21,
        "sigma_OV": sigma_OV,
    }

def normalize_derf_dict(derf_dict):
    out = {}
    for k, v in derf_dict.items():
        out[float(k)] = np.asarray(v, dtype=float)
    return out

def collect_inverse_points_from_fit_bundle(fit_bundle):
    derf_points = defaultdict(list)      # alpha -> list[(layer, value)]
    derf_by_layer = defaultdict(lambda: defaultdict(list))  # alpha -> layer -> [values]
    preln_points = []
    preln_by_layer = defaultdict(list)

    for sample in fit_bundle["samples"]:
        inverse_fit = sample["inverse_fit"]
        vit_points = inverse_fit["vit_points"]
        layers = [int(x) + 1 for x in np.asarray(vit_points["x_shift"], dtype=int)]

        preln_vals = np.asarray(vit_points.get("preln", []), dtype=float)
        for l, v in zip(layers, preln_vals):
            preln_points.append((int(l), float(v)))
            preln_by_layer[int(l)].append(float(v))

        derf_dict = normalize_derf_dict(vit_points.get("derf", {}))
        for alpha, derf_vals in derf_dict.items():
            for l, v in zip(layers, np.asarray(derf_vals, dtype=float)):
                derf_points[float(alpha)].append((int(l), float(v)))
                derf_by_layer[float(alpha)][int(l)].append(float(v))

    derf_by_layer = {
        alpha: {int(layer): np.asarray(vals, dtype=float) for layer, vals in layer_map.items()}
        for alpha, layer_map in derf_by_layer.items()
    }
    preln_by_layer = {int(layer): np.asarray(vals, dtype=float) for layer, vals in preln_by_layer.items()}

    return {
        "derf_points": dict(derf_points),
        "derf_by_layer": derf_by_layer,
        "preln_points": preln_points,
        "preln_by_layer": preln_by_layer,
    }

def mean_curve(layer_to_vals):
    layers = np.asarray(sorted(int(k) for k in layer_to_vals.keys()), dtype=int)
    values = np.asarray([np.mean(layer_to_vals[int(layer)]) for layer in layers], dtype=float)
    return layers, values

def infer_L_inverse(fit_bundle, extracted):
    candidates = []

    if isinstance(fit_bundle, dict):
        if "depth" in fit_bundle:
            try:
                candidates.append(int(fit_bundle["depth"]))
            except Exception:
                pass
        if "num_layers" in fit_bundle:
            try:
                candidates.append(int(fit_bundle["num_layers"]))
            except Exception:
                pass

    if extracted["preln_points"]:
        candidates.append(max(layer for layer, _ in extracted["preln_points"]))

    for alpha, pts in extracted["derf_points"].items():
        if pts:
            candidates.append(max(layer for layer, _ in pts))

    if not candidates:
        raise ValueError("Could not infer L_inverse from the bundle.")

    return max(candidates)

def collect_all_alphas(case_results):
    alpha_set = set()
    for case in case_results:
        alpha_set.update(case["bundle_data"]["derf_points"].keys())
    return sorted(alpha_set)


In [ ]:
# -----------------------------------------------------------------------------
# Per-case analysis
# -----------------------------------------------------------------------------

def compute_case_theory_outputs(case_cfg, bundle_data, L_inverse, q0, p0):
    curve_layers = np.arange(1, int(L_inverse) + 1, dtype=int)

    sigmas = get_case_sigmas(case_cfg["attn_mult"], case_cfg["mlp_mult"])
    sigma_21 = sigmas["sigma_21"]
    sigma_OV = sigmas["sigma_OV"]
    print(sigma_21)
    print(sigma_OV)

    zeta_current = (((sigma_21**2) / 2.0)) / ((((sigma_21**2) / 2.0)) + sigma_OV**2)
    p_star, c_star, roots = find_p_star(sigma_OV=sigma_OV, sigma_21=sigma_21)

    preln_sim = simulate_recursions_full(
        num_layers=L_inverse,
        p0=p0,
        q0=q0,
        n_tokens=N_TOKENS,
        mode="layernorm",
        sigma_w1=sigmas["sigma_w1"],
        sigma_w2=sigmas["sigma_w2"],
        sigma_o=sigmas["sigma_o"],
        sigma_v=sigmas["sigma_v"],
        sigma_a=sigmas["sigma_a"],
    )

    preln_curve = preln_sim["J"][curve_layers]
    preln_asym = (float(L_inverse) / curve_layers) ** zeta_current

    derf_results = {}
    for alpha in sorted(bundle_data["derf_points"].keys()):
        derf_sim = simulate_recursions_full(
            num_layers=L_inverse,
            p0=p0,
            q0=q0,
            n_tokens=N_TOKENS,
            mode="erf",
            alpha=float(alpha),
            sigma_w1=sigmas["sigma_w1"],
            sigma_w2=sigmas["sigma_w2"],
            sigma_o=sigmas["sigma_o"],
            sigma_v=sigmas["sigma_v"],
            sigma_a=sigmas["sigma_a"],
        )

        C_pref = float(alpha) * 2.0 / np.pi
        lam_with_pstar = ((((sigma_21**2) / 2.0) + sigma_OV**2 * p_star) / (C_pref**2 * sigma_21**4))

        derf_J = derf_sim["J"][curve_layers]
        asym_logJ = (np.sqrt(float(L_inverse)) - np.sqrt(curve_layers)) / np.sqrt(lam_with_pstar)
        asym_J = np.exp(np.clip(asym_logJ, np.log(LOG_EPS), 700.0))

        derf_results[float(alpha)] = {
            "J": derf_J,
            "logJ": np.log(np.maximum(derf_J, LOG_EPS)),
            "asym_logJ": asym_logJ,
            "asym_J": asym_J,
            "lambda_with_pstar": lam_with_pstar,
        }

    return {
        "sigmas": sigmas,
        "zeta_current": zeta_current,
        "p_star": p_star,
        "c_star": c_star,
        "roots": roots,
        "curve_layers": curve_layers,
        "preln_curve": preln_curve,
        "preln_asym": preln_asym,
        "derf_results": derf_results,
    }


def analyze_case(case_cfg):
    raw_obj = load_pickle(case_cfg["bundle_path"])
    fit_bundle = unwrap_fit_bundle(raw_obj)
    bundle_data = collect_inverse_points_from_fit_bundle(fit_bundle)

    L_inverse = infer_L_inverse(fit_bundle, bundle_data)

    theory = compute_case_theory_outputs(
        case_cfg=case_cfg,
        bundle_data=bundle_data,
        L_inverse=L_inverse,
        q0=case_cfg["q0"],
        p0=case_cfg["p0"],
    )

    return {
        "case_cfg": case_cfg,
        "fit_bundle": fit_bundle,
        "bundle_data": bundle_data,
        "L_inverse": L_inverse,
        "curve_layers": theory["curve_layers"],
        "sigmas": theory["sigmas"],
        "zeta_current": theory["zeta_current"],
        "p_star": theory["p_star"],
        "c_star": theory["c_star"],
        "roots": theory["roots"],
        "preln_curve": theory["preln_curve"],
        "preln_asym": theory["preln_asym"],
        "derf_results": theory["derf_results"],
    }

case_results = [analyze_case(cfg) for cfg in CASE_CONFIGS]

print("Loaded cases:")
for case in case_results:
    cfg = case["case_cfg"]
    print(
        f"- {cfg['label']}: "
        f"L_inverse={case['L_inverse']}, "
        f"q0={cfg['q0']}, p0={cfg['p0']}, "
        f"alphas={sorted(case['bundle_data']['derf_points'].keys())}"
    )

In [ ]:
# Shared alpha inventory across all cases
ALL_ALPHAS = collect_all_alphas(case_results)
if not ALL_ALPHAS:
    raise ValueError("No derf alpha values were found in the loaded bundles.")

print("Shared derf alpha values:", ALL_ALPHAS)
if len(ALL_ALPHAS) > 1:
    print("Warning: multiple derf alpha values detected; they will be overlaid with the same style.")


In [ ]:
# -----------------------------------------------------------------------------
# Asymptotic heatmap helpers
# -----------------------------------------------------------------------------

WHITE_SOFT_RED_CMAP = LinearSegmentedColormap.from_list(
    "white_soft_red",
    ["#ffffff", "#a50026"],
    N=256,
)
ZETA_GOLD_CMAP = LinearSegmentedColormap.from_list(
    "white_to_classic_gold",
    ["#ffffff", "#d4af37"],
    N=256,
)

HEATMAP_ALPHA = float(ALL_ALPHAS[0])
HEATMAP_SIGMA21_MAX = 8.0
HEATMAP_SIGMAOV_MAX = 8.0
HEATMAP_GRID_SIZE = 20

def compute_asymptotic_heatmap_bundle(alpha, sigma21_max=HEATMAP_SIGMA21_MAX, sigmaOV_max=HEATMAP_SIGMAOV_MAX, grid_size=HEATMAP_GRID_SIZE):
    sigma21_vals = np.linspace(0.0, float(sigma21_max), int(grid_size), dtype=float)
    sigmaOV_vals = np.linspace(0.0, float(sigmaOV_max), int(grid_size), dtype=float)

    lam_inv = np.zeros((sigmaOV_vals.size, sigma21_vals.size), dtype=float)
    zeta = np.zeros_like(lam_inv)

    eps = 1e-8
    C_pref = 2.0 * float(alpha) / np.pi

    for j, sigma_OV in enumerate(sigmaOV_vals):
        for i, sigma_21 in enumerate(sigma21_vals):
            denom = (sigma_21**2) / 2.0 + sigma_OV**2
            zeta[j, i] = ((sigma_21**2) / 2.0) / denom if denom > eps else 0.0

            if sigma_21 <= eps or C_pref <= eps:
                lam_inv[j, i] = 0.0
                continue

            p_star, _, _ = find_p_star(sigma_OV=sigma_OV, sigma_21=sigma_21)
            lam = (((sigma_21**2) / 2.0) + sigma_OV**2 * p_star) / (C_pref**2 * sigma_21**4)
            lam_inv[j, i] = 0.0 if lam <= eps or not np.isfinite(lam) else 1.0 / lam

    return {
        "sigma21_vals": sigma21_vals,
        "sigmaOV_vals": sigmaOV_vals,
        "lam_inv": lam_inv,
        "zeta": zeta,
        "alpha": float(alpha),
    }

asymptotic_heatmap_bundle = compute_asymptotic_heatmap_bundle(alpha=HEATMAP_ALPHA)
print(f"Heatmap alpha (panel c): {HEATMAP_ALPHA}")


In [ ]:
# -----------------------------------------------------------------------------
# Plotting
# -----------------------------------------------------------------------------

DATA_COLOR = "#1f77b4"
THEORY_COLOR = "black"
ASYM_COLOR = "#d62728"


def plot_derf_panel(ax, case, data_color=DATA_COLOR, theory_color=THEORY_COLOR, asym_color=ASYM_COLOR):
    cfg = case["case_cfg"]
    curve_layers = case["curve_layers"]

    for alpha in sorted(case["bundle_data"]["derf_points"].keys()):
        pts = np.asarray(case["bundle_data"]["derf_points"][alpha], dtype=float)
        if pts.size == 0:
            continue
        ax.scatter(
            pts[:, 0],
            np.maximum(pts[:, 1], LOG_EPS),
            s=DERF_POINT_SIZE,
            alpha=DERF_POINT_ALPHA,
            color=data_color,
            edgecolors="none",
            zorder=2,
        )

    for alpha in sorted(case["derf_results"].keys()):
        out = case["derf_results"][alpha]
        ax.plot(
            curve_layers,
            np.maximum(out["J"], LOG_EPS),
            color=theory_color,
            linewidth=DERF_LINEWIDTH,
            zorder=4,
        )
        asym_vals = out.get("asym_J")
        if asym_vals is None:
            asym_vals = np.exp(np.clip(out["asym_logJ"], np.log(LOG_EPS), 700.0))
        ax.plot(
            curve_layers,
            np.maximum(asym_vals, LOG_EPS),
            color=asym_color,
            linewidth=ASYM_LINEWIDTH,
            linestyle=ASYM_LINESTYLE,
            zorder=5,
        )

    ax.set_yscale("log")
    ax.set_title(cfg["label"])
    ax.set_xlabel(r"layer $l$")
    ax.set_ylabel(r"$J^{L,l}$")
    ax.set_xlim(1, case["L_inverse"])
    ax.grid(True, which="major", alpha=0.18)
    ax.grid(False, which="minor")


def plot_preln_panel(ax, case, data_color=DATA_COLOR, theory_color=THEORY_COLOR, asym_color=ASYM_COLOR):
    cfg = case["case_cfg"]
    curve_layers = case["curve_layers"]

    pts = np.asarray(case["bundle_data"]["preln_points"], dtype=float)
    if pts.size > 0:
        ax.scatter(
            pts[:, 0],
            pts[:, 1],
            s=PRELN_POINT_SIZE,
            alpha=PRELN_POINT_ALPHA,
            color=data_color,
            edgecolors="none",
            zorder=2,
        )

    ax.plot(
        curve_layers,
        case["preln_curve"],
        color=theory_color,
        linewidth=PRELN_LINEWIDTH,
        zorder=4,
    )
    ax.plot(
        curve_layers,
        case["preln_asym"],
        color=asym_color,
        linewidth=ASYM_LINEWIDTH,
        linestyle=ASYM_LINESTYLE,
        zorder=5,
    )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(cfg["label"])
    ax.set_xlabel(r"layer $l$")
    ax.set_ylabel(r"$J^{L,l}$")
    ax.set_xlim(1, case["L_inverse"])
    ax.grid(True, which="major", alpha=0.18)
    ax.grid(False, which="minor")


# Map cases into the requested 2x2 order.
case_lookup = {
    (case["case_cfg"]["attn_mult"], case["case_cfg"]["mlp_mult"]): case
    for case in case_results
}
ordered_cases = [
    case_lookup[(2.0, 1.0)],  # upper-left
    case_lookup[(2.0, 2.0)],  # upper-right
    case_lookup[(1.0, 1.0)],  # lower-left
    case_lookup[(1.0, 2.0)],  # lower-right
]

def plot_heatmap_panel(fig, ax, data, sigma21_vals, sigmaOV_vals, cmap, vmin, vmax, cbar_title):
    extent = [
        float(sigma21_vals[0]),
        float(sigma21_vals[-1]),
        float(sigmaOV_vals[0]),
        float(sigmaOV_vals[-1]),
    ]
    im = ax.imshow(
        np.asarray(data, dtype=float),
        origin="lower",
        extent=extent,
        aspect="equal",
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
    )
    ax.set_xlabel(r"$\sigma_{21}$")
    ax.set_ylabel(r"$\sigma_{OV}$")
    ax.set_box_aspect(1)
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.ax.set_title(cbar_title, pad=6.0)
    return im, cbar


fig = plt.figure(figsize=(22, 9))
gs = fig.add_gridspec(
    nrows=2,
    ncols=5,
    width_ratios=[1.0, 1.0, 1.0, 1.0, 1.25],
    wspace=0.30,
    hspace=0.30,
)

# Left side: derf 2x2
ax_d_00 = fig.add_subplot(gs[0, 0])
ax_d_01 = fig.add_subplot(gs[0, 1], sharex=ax_d_00, sharey=ax_d_00)
ax_d_10 = fig.add_subplot(gs[1, 0], sharex=ax_d_00, sharey=ax_d_00)
ax_d_11 = fig.add_subplot(gs[1, 1], sharex=ax_d_00, sharey=ax_d_00)

# Middle: pre-LN 2x2
ax_p_00 = fig.add_subplot(gs[0, 2])
ax_p_01 = fig.add_subplot(gs[0, 3], sharex=ax_p_00, sharey=ax_p_00)
ax_p_10 = fig.add_subplot(gs[1, 2], sharex=ax_p_00, sharey=ax_p_00)
ax_p_11 = fig.add_subplot(gs[1, 3], sharex=ax_p_00, sharey=ax_p_00)

# Right column: heatmaps
ax_c = fig.add_subplot(gs[0, 4])
ax_d = fig.add_subplot(gs[1, 4])

derf_axes = [ax_d_00, ax_d_01, ax_d_10, ax_d_11]
preln_axes = [ax_p_00, ax_p_01, ax_p_10, ax_p_11]

for ax, case in zip(derf_axes, ordered_cases):
    plot_derf_panel(ax, case)

for ax, case in zip(preln_axes, ordered_cases):
    plot_preln_panel(ax, case)

lam_data = np.asarray(asymptotic_heatmap_bundle["lam_inv"], dtype=float)
plot_heatmap_panel(
    fig,
    ax_c,
    data=lam_data,
    sigma21_vals=asymptotic_heatmap_bundle["sigma21_vals"],
    sigmaOV_vals=asymptotic_heatmap_bundle["sigmaOV_vals"],
    cmap=WHITE_SOFT_RED_CMAP,
    vmin=0.0,
    vmax=max(float(np.nanmax(lam_data)), 1e-6),
    cbar_title=r"$\lambda^{-1}$",
)
plot_heatmap_panel(
    fig,
    ax_d,
    data=asymptotic_heatmap_bundle["zeta"],
    sigma21_vals=asymptotic_heatmap_bundle["sigma21_vals"],
    sigmaOV_vals=asymptotic_heatmap_bundle["sigmaOV_vals"],
    cmap=ZETA_GOLD_CMAP,
    vmin=0.0,
    vmax=1.0,
    cbar_title=r"$\zeta$",
)

for ax in [ax_d_01, ax_d_11, ax_p_01, ax_p_11]:
    ax.set_ylabel("")
for ax in [ax_d_00, ax_d_01, ax_p_00, ax_p_01]:
    ax.set_xlabel("")

fig.text(0.21, 0.98, "(a) Inverse derf", ha="center", va="top", fontsize=18)
fig.text(0.59, 0.98, "(b) Inverse pre-LN", ha="center", va="top", fontsize=18)
fig.text(0.865, 0.98, rf"(c) $\lambda^{{-1}}$ for $\alpha={HEATMAP_ALPHA:g}$", ha="center", va="top", fontsize=18)
fig.text(0.865, 0.49, r"(d) $\zeta$", ha="center", va="top", fontsize=18)

legend_handles = [
    plt.Line2D([0], [0], color=THEORY_COLOR, linewidth=DERF_LINEWIDTH, label="inverse APJN simulation"),
    plt.Line2D([0], [0], color=ASYM_COLOR, linewidth=ASYM_LINEWIDTH, linestyle=ASYM_LINESTYLE, label="asymptotic"),
    plt.Line2D([0], [0], marker="o", linestyle="None", color=DATA_COLOR, alpha=DERF_POINT_ALPHA, label="bundle data"),
]

ax_d_00.legend(handles=legend_handles, frameon=False, loc="upper right")
ax_p_00.legend(handles=legend_handles, frameon=False, loc="upper right")

fig.suptitle("Inverse APJN bundle overlays across attention / MLP multipliers", y=1.03, fontsize=20)

if SAVE_FIGURE:
    fig.savefig(FIGURE_PATH, bbox_inches="tight")
    print(f"Saved figure to: {FIGURE_PATH}")

plt.show()


In [ ]:
# -----------------------------------------------------------------------------
# Optional summary table
# -----------------------------------------------------------------------------

for case in case_results:
    cfg = case["case_cfg"]
    print("=" * 90)
    print(cfg["label"])
    print(f"bundle_path = {cfg['bundle_path']}")
    print(f"q0 = {cfg['q0']}, p0 = {cfg['p0']}")
    print(f"L_inverse = {case['L_inverse']}")
    print(f"p_star = {case['p_star']:.8f}")
    print(f"zeta_current = {case['zeta_current']:.8f}")
    print(f"alphas = {sorted(case['derf_results'].keys())}")


In [ ]:
# -----------------------------------------------------------------------------
# Fast search for the shared (q0, p0) that minimizes MAPE for each case
# -----------------------------------------------------------------------------
# Notes:
# - MAPE is computed against ALL raw bundle points, matched at their corresponding
#   layer indices, for both pre-LN and inverse derf.
# - The same (q0, p0) is shared between pre-LN and inverse derf within a case.
# - The (q0, rho0) candidate grid is shared, but the APJN values themselves are
#   case-specific because the attention / MLP multipliers change the sigmas.
# - This version is much faster than the original per-candidate loop because it
#   evaluates the whole (q0, rho0) grid in vectorized batches.

OPT_Q0_BOUNDS = (0.05, 20)      # search bounds for q0
OPT_RHO_BOUNDS = (0.0, 0.99)    # rho0 = p0 / q0
OPT_Q0_GRID_SIZE = 100           # coarse grid size in q0
OPT_RHO_GRID_SIZE = 100         # coarse grid size in rho0
OPT_REFINEMENT_STEPS = 3         # 1 = initial grid only
OPT_REFINEMENT_SHRINK = 0.35     # search-window shrink factor after each refinement


def positive_grid(lo, hi, n):
    if lo <= 0:
        return np.linspace(lo, hi, n)
    return np.exp(np.linspace(np.log(lo), np.log(hi), n))


_GRID_CACHE = {}


def build_candidate_grid(q_lo, q_hi, rho_lo, rho_hi, q0_grid_size, rho_grid_size):
    key = (
        float(q_lo), float(q_hi),
        float(rho_lo), float(rho_hi),
        int(q0_grid_size), int(rho_grid_size),
    )
    if key not in _GRID_CACHE:
        q_grid = positive_grid(q_lo, q_hi, q0_grid_size)
        rho_grid = np.linspace(rho_lo, rho_hi, rho_grid_size)
        q_mesh, rho_mesh = np.meshgrid(q_grid, rho_grid, indexing="ij")
        _GRID_CACHE[key] = (
            q_grid,
            rho_grid,
            q_mesh.ravel(),
            rho_mesh.ravel(),
        )
    return _GRID_CACHE[key]


def build_optimizer_obs(points):
    pts = np.asarray(points, dtype=float)
    if pts.size == 0:
        return {
            "layer_idx": np.asarray([], dtype=int),
            "obs": np.asarray([], dtype=float),
            "inv_abs_obs": np.asarray([], dtype=float),
            "count": 0,
        }

    if pts.ndim != 2 or pts.shape[1] != 2:
        raise ValueError("points must be an array-like of shape (N, 2).")

    layers = pts[:, 0].astype(int)
    obs = pts[:, 1].astype(float)
    mask = np.isfinite(layers) & np.isfinite(obs) & (np.abs(obs) > 1e-12)

    layers = layers[mask]
    obs = obs[mask]

    return {
        "layer_idx": layers - 1,   # theory_curve is indexed by layer-1
        "obs": obs,
        "inv_abs_obs": 1.0 / np.abs(obs),
        "count": int(obs.size),
    }


def prepare_case_optimizer_payload(case):
    cfg = case["case_cfg"]
    sigmas = get_case_sigmas(cfg["attn_mult"], cfg["mlp_mult"])
    sigma_21 = sigmas["sigma_21"]
    sigma_OV = sigmas["sigma_OV"]
    p_star, c_star, roots = find_p_star(sigma_OV=sigma_OV, sigma_21=sigma_21)

    derf_alphas = sorted(case["bundle_data"]["derf_points"].keys())

    return {
        "case_cfg": cfg,
        "L_inverse": int(case["L_inverse"]),
        "sigmas": sigmas,
        "sigma_21": sigma_21,
        "sigma_OV": sigma_OV,
        "p_star": p_star,
        "c_star": c_star,
        "roots": roots,
        "derf_alphas": derf_alphas,
        "preln_obs": build_optimizer_obs(case["bundle_data"]["preln_points"]),
        "derf_obs": {
            float(alpha): build_optimizer_obs(case["bundle_data"]["derf_points"].get(alpha, []))
            for alpha in derf_alphas
        },
    }


def simulate_inverse_curve_batch(
    num_layers,
    q0_values,
    p0_values,
    mode,
    alpha=1.0,
    sigma_w1=0.64,
    sigma_w2=1.28,
    sigma_o=0.64,
    sigma_v=0.64,
    sigma_a=0.64 * 0.64,
):
    L = int(num_layers)
    q = np.asarray(q0_values, dtype=float).copy()
    p = np.asarray(p0_values, dtype=float).copy()

    if q.shape != p.shape:
        raise ValueError("q0_values and p0_values must have the same shape.")

    n_candidates = q.size
    eps = 1e-12
    chi = np.empty((n_candidates, L), dtype=float)

    att_scale = (sigma_o ** 2) * (sigma_v ** 2)
    mlp_scale = (sigma_w1 ** 2) * (sigma_w2 ** 2)

    mode_l = mode.lower()
    if mode_l == "erf":
        alpha = float(alpha)
        alpha2 = alpha ** 2

        for l in range(L):
            up = tilde_p_erf_np(q, p, alpha)

            qh = q + att_scale * up
            ph = p + att_scale * up

            chi[:, l] = 1.0 + mlp_scale * (2.0 * alpha2 / np.pi) * (
                1.0 / np.sqrt(1.0 + 4.0 * alpha2 * qh)
            )

            u_half = tilde_q_erf_np(qh, alpha)
            v_half = tilde_p_erf_np(qh, ph, alpha)
            rho_half = np.clip(v_half / (u_half + eps), -1.0, 1.0)

            q = qh + 0.5 * mlp_scale * u_half
            p = ph + mlp_scale * u_half * kappa_relu_np(rho_half)

    elif mode_l == "layernorm":
        for l in range(L):
            rho = np.clip(p / (q + eps), -1.0, 1.0)

            qh = q + att_scale * rho
            ph = p + att_scale * rho

            chi[:, l] = 1.0 + mlp_scale / (2.0 * (qh + eps))

            rho_half = np.clip(ph / (qh + eps), -1.0, 1.0)
            q = qh + 0.5 * mlp_scale
            p = ph + mlp_scale * kappa_relu_np(rho_half)
    else:
        raise ValueError("mode must be 'erf' or 'layernorm'")

    suffix_products = np.flip(
        np.cumprod(np.flip(chi, axis=1), axis=1),
        axis=1,
    )

    theory_curve = np.empty((n_candidates, L), dtype=float)
    if L > 1:
        theory_curve[:, :-1] = suffix_products[:, 1:]
    theory_curve[:, -1] = 1.0
    return theory_curve


def batched_relative_error_sum(curve_matrix, obs_block):
    count = int(obs_block["count"])
    if count == 0:
        return np.zeros(curve_matrix.shape[0], dtype=float), 0

    preds = curve_matrix[:, obs_block["layer_idx"]]
    obs = obs_block["obs"][None, :]
    inv_abs_obs = obs_block["inv_abs_obs"][None, :]
    err_sum = np.abs(preds - obs) * inv_abs_obs
    return np.sum(err_sum, axis=1), count


def batched_shared_mape_for_case(payload, q0_values, p0_values):
    sigmas = payload["sigmas"]
    L_inverse = payload["L_inverse"]

    total_err = np.zeros(np.asarray(q0_values).size, dtype=float)
    total_count = 0

    preln_curve = simulate_inverse_curve_batch(
        num_layers=L_inverse,
        q0_values=q0_values,
        p0_values=p0_values,
        mode="layernorm",
        sigma_w1=sigmas["sigma_w1"],
        sigma_w2=sigmas["sigma_w2"],
        sigma_o=sigmas["sigma_o"],
        sigma_v=sigmas["sigma_v"],
        sigma_a=sigmas["sigma_a"],
    )
    err_sum, count = batched_relative_error_sum(preln_curve, payload["preln_obs"])
    total_err += err_sum
    total_count += count

    for alpha in payload["derf_alphas"]:
        derf_curve = simulate_inverse_curve_batch(
            num_layers=L_inverse,
            q0_values=q0_values,
            p0_values=p0_values,
            mode="erf",
            alpha=float(alpha),
            sigma_w1=sigmas["sigma_w1"],
            sigma_w2=sigmas["sigma_w2"],
            sigma_o=sigmas["sigma_o"],
            sigma_v=sigmas["sigma_v"],
            sigma_a=sigmas["sigma_a"],
        )
        err_sum, count = batched_relative_error_sum(derf_curve, payload["derf_obs"][float(alpha)])
        total_err += err_sum
        total_count += count

    if total_count == 0:
        return np.full_like(total_err, np.inf, dtype=float)

    return total_err / float(total_count)


def optimize_q0_p0_for_case(
    case,
    q0_bounds=OPT_Q0_BOUNDS,
    rho_bounds=OPT_RHO_BOUNDS,
    q0_grid_size=OPT_Q0_GRID_SIZE,
    rho_grid_size=OPT_RHO_GRID_SIZE,
    refinement_steps=OPT_REFINEMENT_STEPS,
    refinement_shrink=OPT_REFINEMENT_SHRINK,
):
    payload = prepare_case_optimizer_payload(case)

    q_lo, q_hi = map(float, q0_bounds)
    rho_lo, rho_hi = map(float, rho_bounds)

    best = None

    for step in range(refinement_steps):
        q_grid, rho_grid, q0_flat, rho0_flat = build_candidate_grid(
            q_lo, q_hi, rho_lo, rho_hi, q0_grid_size, rho_grid_size
        )
        p0_flat = q0_flat * rho0_flat

        mape_flat = batched_shared_mape_for_case(payload, q0_flat, p0_flat)
        best_idx = int(np.argmin(mape_flat))

        best = {
            "mape": float(mape_flat[best_idx]),
            "q0": float(q0_flat[best_idx]),
            "p0": float(p0_flat[best_idx]),
            "rho0": float(rho0_flat[best_idx]),
            "step": step,
        }

        if step < refinement_steps - 1:
            log_q_best = np.log(max(best["q0"], 1e-12))
            half_log_width = 0.5 * (np.log(q_hi) - np.log(q_lo)) * refinement_shrink
            q_lo = max(1e-8, float(np.exp(log_q_best - half_log_width)))
            q_hi = float(np.exp(log_q_best + half_log_width))

            rho_half_width = 0.5 * (rho_hi - rho_lo) * refinement_shrink
            rho_lo = max(0.0, best["rho0"] - rho_half_width)
            rho_hi = min(0.999999, best["rho0"] + rho_half_width)

    best["theory"] = compute_case_theory_outputs(
        case_cfg=case["case_cfg"],
        bundle_data=case["bundle_data"],
        L_inverse=case["L_inverse"],
        q0=best["q0"],
        p0=best["p0"],
    )
    best["payload"] = payload
    return best


optimized_case_results = []
for case in ordered_cases:
    best = optimize_q0_p0_for_case(case)
    optimized_case_results.append({
        "case": case,
        "best": best,
    })

print("Optimized shared (q0, p0) per case")
print("=" * 100)
for item in optimized_case_results:
    case = item["case"]
    best = item["best"]
    cfg = case["case_cfg"]
    print(cfg["label"])
    print(f"  bundle_path = {cfg['bundle_path']}")
    print(f"  best q0     = {best['q0']:.8f}")
    print(f"  best p0     = {best['p0']:.8f}")
    print(f"  best rho0   = {best['rho0']:.8f}")
    print(f"  shared MAPE = {100.0 * best['mape']:.4f}%")
    print(f"  original    = (q0={cfg['q0']:.8f}, p0={cfg['p0']:.8f})")
    print("-" * 100)
